In [ ]:
import pandas as pd
import re

# 1. Carregar o dataset (caminho relativo limpo)
df = pd.read_csv('../data/olist_order_reviews_dataset.csv')

# 2. Remover linhas onde o comentário está vazio
df_clean = df.dropna(subset=['review_comment_message']).copy()

# 3. Criar as categorias de sentimento
def create_sentiment(score):
    if score >= 4:
        return 'Positive'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Negative'

df_clean['sentiment'] = df_clean['review_score'].apply(create_sentiment)

# 4. Limpar o texto
def clean_text(text):
    text = str(text).lower()  # Minúsculo
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)  # Remove URLs
    text = re.sub(r'[^a-zà-ÿ\s]', '', text)  # Remove pontuação e números
    text = re.sub(r'\s+', ' ', text).strip()  # Remove espaços extras
    return text

df_clean['clean_text'] = df_clean['review_comment_message'].apply(clean_text)

# 5. Mostrar o resultado
print("Shape original:", df.shape)
print("Shape após remover nulos:", df_clean.shape)
print("\nDistribuição dos sentimentos:")
print(df_clean['sentiment'].value_counts())

print("\n--- Exemplo de texto limpo ---")
print("Antes:", df_clean['review_comment_message'].iloc[0])
print("Depois:", df_clean['clean_text'].iloc[0])

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Remover comentários duplicados para evitar data leakage
# (mantendo apenas a primeira ocorrência de cada texto limpo)
df_final = df_clean.drop_duplicates(subset=['clean_text']).copy()
print(f"Comentários após remover duplicatas: {len(df_final)}")

# 2. Separar o X (texto) e o y (sentimento)
X = df_final['clean_text'].tolist()
y = df_final['sentiment'].tolist()

# 3. Primeiro split: Separar 80% para treino+validação e 20% para teste
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Segundo split: Separar o treino+validação em 80% treino e 20% validação
# (Isso dá 64% treino, 16% validação e 20% teste do total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

# 5. Verificar os tamanhos
print(f"Treino: {len(X_train)}")
print(f"Validação: {len(X_val)}")
print(f"Teste: {len(X_test)}")

In [ ]:
# Criar DataFrames para salvar
df_train = pd.DataFrame({'text': X_train, 'label': y_train})
df_val = pd.DataFrame({'text': X_val, 'label': y_val})
df_test = pd.DataFrame({'text': X_test, 'label': y_test})

# Salvar na pasta data/
df_train.to_csv('../data/train.csv', index=False)
df_val.to_csv('../data/val.csv', index=False)
df_test.to_csv('../data/test.csv', index=False)

print("Arquivos train.csv, val.csv e test.csv salvos na pasta data/")